# 🎙️ MirZa SaQib — AI Voice Generator

> ⚠️ **Pehle:**  →  → **T4 GPU**
>
> Phir **Cell 1** chalao email check karo, phir baaki cells ✅

In [ ]:
import urllib.request, json, sys
import ipywidgets as widgets
from IPython.display import display, HTML

email_box = widgets.Text(placeholder="apni_email@gmail.com", description="📧 Email:", layout=widgets.Layout(width="400px"))
btn = widgets.Button(description="Check Access ✅", button_style="success", layout=widgets.Layout(width="180px"))
out = widgets.Output()

def check(b):
    out.clear_output()
    with out:
        em = email_box.value.strip().lower()
        if not em or "@" not in em:
            print("⚠️ Sahi email daalo!"); return
        try:
            sid = "13GxCB3yUkF3RrsBrVh2vqImSBgkjBc2SWA08Qzf21tg"
            url = f"https://docs.google.com/spreadsheets/d/{sid}/gviz/tq?tqx=out:json&sheet=Sheet1"
            with urllib.request.urlopen(url, timeout=10) as r:
                raw = r.read().decode()
                raw = raw[raw.index("{"):raw.rindex("}")+1]
                data = json.loads(raw)
            rows = data["table"]["rows"]
            sc = rows[0]["c"][1]
            if not (sc and sc.get("v","").strip().upper() == "ON"):
                print("🔴 Tool abhi band hai. Baad mein try karo."); return
            allowed = {row["c"][0]["v"].strip().lower() for row in rows[1:] if row["c"][0] and row["c"][0].get("v")}
            import builtins
            if em in allowed:
                builtins._ok = True
                print("✅ Access Granted! Welcome 🎉 — Ab neeche Install cell chalao 👇")
            else:
                builtins._ok = False
                print("❌ Access Denied! MirZa SaQib se rabta karo.")
        except Exception as e:
            print(f"⚠️ Error: {e}")

btn.on_click(check)
display(email_box, btn, out)

# OmniVoice Quick Start

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/k2-fsa/OmniVoice/blob/master/docs/OmniVoice.ipynb)

This notebook demonstrates the basic usage of [OmniVoice](https://github.com/k2-fsa/OmniVoice), a massively multilingual zero-shot TTS model supporting 600+ languages.

**Contents:**
1. Installation
2. Option A — Gradio Demo (interactive web UI, no code needed)
3. Option B — Python API
   - 3.1 Load Model
   - 3.2 Voice Cloning
   - 3.3 Voice Design
   - 3.4 Auto Voice

## 1. Installation

Colab already provides a compatible PyTorch + CUDA environment, so we only need to install OmniVoice.

In [ ]:
import builtins, sys
if not getattr(builtins, "_ok", False):
    print("❌ Pehle upar email check karo!"); sys.exit()
!pip install git+https://github.com/mirzasaqibusa-oss/Mirza-tools.git -q
print("✅ Install complete!")

## 2. Option A — Gradio Demo

Launch an interactive web UI with a public Gradio link. The `--share` flag creates a temporary public URL so you can access the demo from any browser.

> **If you prefer to use the Python API directly, skip to Option B below.**

In [ ]:
!omnivoice-demo --share

## 3. Option B — Python API

### 3.1 Load Model

In [ ]:
from omnivoice import OmniVoice
import soundfile as sf
import torch
from IPython.display import Audio, display

model = OmniVoice.from_pretrained(
    "k2-fsa/OmniVoice",
    device_map="cuda:0",
    dtype=torch.float16,
    load_asr=True,
)

### 3.2 Voice Cloning

Clone a voice from a short (3-10s) reference audio clip. Upload your own `ref.wav` or use any audio file.

`ref_text` is optional — if omitted, the model uses Whisper ASR to auto-transcribe it.

In [ ]:
from google.colab import files

print("Upload a reference audio file (wav/mp3/flac):")
uploaded = files.upload()
ref_audio_path = list(uploaded.keys())[0]
print(f"Uploaded: {ref_audio_path}")

In [ ]:
audio = model.generate(
    text="Hello, this is a test of zero-shot voice cloning.",
    ref_audio=ref_audio_path,
    # ref_text="Transcription of the reference audio.",  # optional
)

sf.write("clone_out.wav", audio[0], 24000)
display(Audio(audio[0], rate=24000))

### 3.3 Voice Design

Describe the desired voice with speaker attributes — no reference audio needed.

Supported attributes: gender, age, pitch, style (whisper), English accent, Chinese dialect. See [docs/voice-design.md](https://github.com/k2-fsa/OmniVoice/blob/master/docs/voice-design.md) for the full list.

In [ ]:
audio = model.generate(
    text="Hello, this is a test of zero-shot voice design.",
    instruct="female, low pitch, british accent",
)

sf.write("design_out.wav", audio[0], 24000)
display(Audio(audio[0], rate=24000))

### 3.4 Auto Voice

Let the model choose a voice automatically — no reference audio or instruct needed.

In [ ]:
audio = model.generate(
    text="This is a sentence generated with automatic voice selection.",
)

sf.write("auto_out.wav", audio[0], 24000)
display(Audio(audio[0], rate=24000))